# CryptoGraphic Malware Detection

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import numpy as np
import hashlib
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import re
import os
import struct

class CryptoMalwareDetector:
    def __init__(self):
        self.rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
        self.vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,3))
        self.entropy_threshold = 7.5

    def calculate_entropy(self, data):
        byte_counts = np.bincount(np.frombuffer(data, dtype=np.uint8))
        probs = byte_counts / len(data)
        return -np.sum(probs * np.log2(probs + 1e-10))

    def extract_pe_features(self, pe_data):
        features = []
        if len(pe_data) > 64:
            dos_header = struct.unpack('<H', pe_data[0:2])[0]
            features.append(dos_header)
            pe_offset = struct.unpack('<L', pe_data[60:64])[0]
            if pe_offset < len(pe_data) - 24:
                machine_type = struct.unpack('<H', pe_data[pe_offset+4:pe_offset+6])[0]
                features.extend([machine_type, len(pe_data)])
        return features + [0] * max(0, 10 - len(features))

    def analyze_strings(self, binary_data):
        strings = re.findall(rb'[A-Za-z0-9/\-:.,;_$&@=?+]{4,}', binary_data)
        crypto_indicators = [b'AES', b'RSA', b'encrypt', b'decrypt', b'cipher', b'key']
        return sum(1 for s in strings if any(ind in s for ind in crypto_indicators))

    def extract_features(self, file_path):
        with open(file_path, 'rb') as f:
            data = f.read()
        entropy = self.calculate_entropy(data)
        pe_features = self.extract_pe_features(data)
        crypto_strings = self.analyze_strings(data)
        file_size = len(data)
        return [entropy, file_size, crypto_strings] + pe_features

    def train(self, malware_dir, benign_dir):
        X, y = [], []
        for file_name in os.listdir(malware_dir):
            if os.path.isfile(os.path.join(malware_dir, file_name)):
                X.append(self.extract_features(os.path.join(malware_dir, file_name)))
                y.append(1)
        for file_name in os.listdir(benign_dir):
            if os.path.isfile(os.path.join(benign_dir, file_name)):
                X.append(self.extract_features(os.path.join(benign_dir, file_name)))
                y.append(0)
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
        self.rf_model.fit(X_train, y_train)
        return self.rf_model.score(X_test, y_test)

    def detect_ransomware(self, file_path):
        features = self.extract_features(file_path)
        prediction = self.rf_model.predict([features])[0]
        confidence = self.rf_model.predict_proba([features])[0][prediction]
        return prediction == 1, confidence